In [ ]:
# Investigate League
import os

import dotenv
import pandas as pd
import yahoo_fantasy_api as yfa
from openai import OpenAI

dotenv.load_dotenv()

# Load Session Context
sc = yfa.OAuth2Manager.get_oauth2()


In [ ]:


game_code = "nfl"
year = 2024
gm = yfa.Game(sc, game_code)
lg_id = gm.league_ids(year=year)[0]
lg = yfa.League(sc, lg_id)
standings = lg.standings()
settings = lg.settings()


In [ ]:
teams_data = lg.teams()

# Store team manager nicknames for use in other cells
team_managers = []
team_data = {}

for id, tdata in teams_data.items():
    # Correct way to access manager nickname
    manager_info = tdata.get("managers", [{}])[0]
    nickname = manager_info.get("manager", {}).get("nickname", "No Manager")
    team_name = tdata["name"]

    team_managers.append(nickname)
    team_data[id] = {"team_name": team_name, "manager_nickname": nickname}

    print(f"{team_name} - {nickname}")

print(f"\nExtracted {len(team_managers)} manager nicknames:")
print(team_managers)

In [ ]:
# Get standings and map team names to manager nicknames
standings_data = lg.standings()

# Create a mapping from team name to manager nickname using current teams data
team_name_to_manager = {}
for _, team_info in team_data.items():
    team_name_to_manager[team_info["team_name"]] = team_info["manager_nickname"]

print("Team name to manager mapping:")
for team_name, manager in team_name_to_manager.items():
    print(f"  {team_name} -> {manager}")

print("\nStandings with manager nicknames:")
print("=" * 50)

# Enhanced standings with manager nicknames
standings_with_managers = []
for team_standing in standings_data:
    team_name = team_standing["name"]
    manager_nickname = team_name_to_manager.get(team_name, "Unknown")

    # Create enhanced standing record
    enhanced_standing = team_standing.copy()
    enhanced_standing["manager_nickname"] = manager_nickname
    standings_with_managers.append(enhanced_standing)

    print(f"Rank {team_standing['rank']}: {manager_nickname} ({team_name})")

# Store for use in other cells
standings_with_manager_names = standings_with_managers

In [ ]:
# Function to map team name to manager nickname
def map_team_to_manager(team_name, current_mapping):
    """
    Map team name to manager nickname, only exact matches on team names
    """
    return current_mapping.get(team_name, "Unknown")


years = [2020, 2021, 2022, 2023, 2024]
names = ["Super spreaders", "A Scene Like No Other"]
data = {}

for year in years:
    leagues = gm.league_ids(year=year)
    for league in leagues:
        league = yfa.League(sc, league)
        if league.settings()["name"] in names:
            print("Found: " + league.settings()["name"] + " in " + str(year))

            # Get standings and map to manager nicknames
            historical_standings = league.standings()
            standings_with_managers = []

            for team_standing in historical_standings:
                team_name = team_standing["name"]
                manager_nickname = map_team_to_manager(team_name, team_name_to_manager)

                # Create enhanced standing record with manager nickname
                enhanced_standing = team_standing.copy()
                enhanced_standing["manager_nickname"] = manager_nickname
                enhanced_standing["original_team_name"] = team_name
                standings_with_managers.append(enhanced_standing)

                print(f"  {year}: {team_name} -> {manager_nickname}")

            data[year] = {
                "leaguename": league.settings()["name"],
                "standings": standings_with_managers,
            }
    print(f"Completed year {year}\n")

In [ ]:
names = ["Super spreaders", "A Scene Like No Other"]
data = {}
for year in years:
    leagues = gm.league_ids(year=year)
    for l in leagues:
        league = yfa.League(sc, l)
        if league.settings()["name"] in names:
            print("Found: " + league.settings()["name"] + " in " + str(year))
            data[year] = {
                "leaguename": league.settings()["name"],
                # "settings": l.settings(),
                "standings": league.standings(),
            }
    # print(year, gm.league_ids(year=year))


In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"Use the following league results to estimate fantasy league standings for a 12 team league 2025 season with an explanation for each team."
            f"Here are the league results for years:{list(data.keys())} " + str(data),
        }
    ],
)
response.choices[0].message.content

In [ ]:
positions = lg.positions()
for pos in [p for p in positions.keys() if p not in ["W/R/T"]]:
    freeagents = lg.free_agents(pos)
    print(f"Free agents for position {pos}: {freeagents}")


In [ ]:
qb_freeagents = lg.free_agents("QB")
qb_df = pd.DataFrame(qb_freeagents)
qb_df.sort_values(by="percent_owned", ascending=False, inplace=True)
qb_df.reset_index(drop=True, inplace=True)
# qb_df.head(5)
for index, row in qb_df.head(2).iterrows():
    details = lg.player_details(row["player_id"])
    print(f"Player: {row['name']}, Owned: {row['percent_owned']}%, Details: {details}")

In [ ]:
# Collect Sachin's teams and rosters over the years
target_manager = "Sachin"
sachin_data = {}

print(f"Collecting data for {target_manager}:")
print("=" * 50)

for year in years:
    leagues = gm.league_ids(year=year)
    for league in leagues:
        league = yfa.League(sc, league)
        if league.settings()["name"] in names:
            print(f"\nChecking {year} - {league.settings()['name']}")

            # Get all teams for this year
            year_teams = league.teams()

            # Find Sachin's team
            sachin_team_key = None
            sachin_team_name = None

            for team_id, team_info in year_teams.items():
                # Get manager info
                manager_info = team_info.get("managers", [{}])[0]
                manager_nickname = manager_info.get("manager", {}).get(
                    "nickname", "No Manager"
                )

                if manager_nickname == target_manager:
                    sachin_team_key = team_info["team_key"]
                    sachin_team_name = team_info["name"]
                    print(
                        f"  Found {target_manager}'s team: {sachin_team_name} (key: {sachin_team_key})"
                    )
                    break

            if sachin_team_key:
                # Get the roster using yfa.Team()
                try:
                    team = yfa.Team(sc, sachin_team_key)
                    roster = team.roster()

                    sachin_data[year] = {
                        "team_key": sachin_team_key,
                        "team_name": sachin_team_name,
                        "roster": roster,
                        "league_name": league.settings()["name"],
                    }

                    print(f"  Roster collected: {len(roster)} players")

                    # Show a few player names for verification
                    if roster:
                        player_names = [
                            player.get("name", "Unknown") for player in roster[:3]
                        ]
                        print(f"  Sample players: {', '.join(player_names)}...")

                except Exception as e:
                    print(f"  Error getting roster: {e}")
            else:
                print(f"  {target_manager} not found in {year}")

print(
    f"\nCollected data for {target_manager} across {len(sachin_data)} years: {list(sachin_data.keys())}"
)

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"Provide me a fantasy football roster analysis for Sachin based on the following rosters from different years. "
            f"### Rosters:\n"
            f"{sachin_data}",
        }
    ],
)


In [ ]:
print(response.choices[0].message.content)